<img src="logo.png" width="100" height="100" align="right"> 

## AutoMoL

Pipeline for automated machine learning for drug design.

* **Authors**: Mazen Ahmad, Joris Tavernier, Natalia Dyubankova, Marvin Steijaert
* **Contact**: joris.tavernier@openanalytics.eu, Marvin.Steijaert@openanalytics.eu


&copy; All rights reserved, Open Analytics NV, 2021-2025.

### General Usage Guidelines

We assume that the data provided to the pipeline is clean data. The pipeline **does not handle qualifiers or outliers** in the property target. This is the responsibility of the end-user. This should be done with care and with knowledge of the data at hand. 

### Ignoring all warnings

mainly disables convergence warnings, remove if you would like to get python warnings

In [ ]:
import warnings
import sys, os

if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore" # Also affects subprocesses

## Stacking/Blending Classifier
#### Generating Bottleneck features and combining different shallow estimators.
In this notebook we will show how to perform classification for a data set from Therapeutics Data Commons (https://tdcommons.ai/overview). The notebook works with a Dataframe and can easily adopted by reading from csv files. 

This notebooks turn regression into classification and assigns classes based on provided values. 

Each cell starts with the parameters to be adjusted to your specific case and these parameters are surrounded by *#####################################################*. This is followed by import statement and the code will be executed in that cell. Each cell is accompanied with a markdown cell detailing the goal of the cell.

### Reading the data 

The dataset is downloaded using the functionality from PyTDC and returns two pandas dataframes (train & test), the variable **name** defines which dataset, the column in the file containing the smiles is set in the variable **smiles_column**. The smiles are then standardized and placed in the column provided in the variable **standard_smiles_column**. If you want to add rdkit descriptors to the feature set, set **check_rdkit_desc** which excludes smiles with nan descriptors.

The **verbose** variable can be set to *1* when more output is desired. 

In [ ]:
#####################################################
#name of the dataset
#name='Caco2_Wang'
#name='HydrationFreeEnergy_FreeSolv'
name='Lipophilicity_AstraZeneca'
#name='PPBR_AZ'
#set the column in the data file where the smiles can be found
smiles_column='Drug'
#verbosity
verbose=1
# set to true if you want to add rdkit descriptors to the feature set
check_rdkit_desc=False
#name of the new column to add the standardized smiles in the dataframe (choice is free)
standard_smiles_column='stand_SMILES'
#####################################################
from tdc.single_pred import ADME
import numpy as np, pandas as pd
from  matplotlib import pyplot as plt
from automol.property_prep import add_stereo_smiles,validate_rdkit_smiles, add_rdkit_standardized_smiles
#dictionary to keep track of the relevant training parameters

data = ADME(name = name)
split = data.get_split()
df=split['train']
df['Y_multi']=df['Y']
test_df=split['test']
test_df['Y_multi']=test_df['Y']

model_param={}
model_param['Data file']=name

#in case of reading data from csv
#df= pd.read_csv(file_name, na_values = ['NAN', '?','NaN'])

add_rdkit_standardized_smiles(df, smiles_column,verbose=verbose,outname=standard_smiles_column)
add_rdkit_standardized_smiles(test_df, smiles_column,verbose=verbose,outname=standard_smiles_column)

model_param['Standardization']='rdkit'

if check_rdkit_desc: 
    validate_rdkit_smiles(df, standard_smiles_column,verbose=verbose)

df.dropna(inplace=True, subset = [standard_smiles_column])
df.head(5)

### Building the classes in the case of continuous data
The variable **properties** contains a list of strings. This list corresponds to columns in the data file. These columns are the properties required to be fitted. 

The variable **nb_classes** contains for each property the number of classes and the variable **class_values** contains for each property the splits between classes. In this case the first property is split in classes: <=1 and >1. The second property is split in three classes: <=1; >1 & <=5; and >5.  

If the actual values are unknown, it is possible to set the variables **class_quantiles** as the a list of quantiles for each class. 

The last variable is **min_allowed_class_samples** is the minimum number of samples allowed in a class. If this condition is not met, classes will be based on generated quantiles.

In the case of categorical properties, set **categorical** to True.

In [ ]:
#####################################################
#set the properties to be fitted
categorical=False
properties=['Y','Y_multi']
#total nb of classes for each property
nb_classes=[2,3]
#set class values for classification
#provide values for splits between classes, e.g. 3 classes, provide 2 values 
#list of values to differentiate between properties
class_values= [[2],[2,3]]
#set for the use of relative volume values (quantiles)
class_quantiles=None #[ [0.2, 0.4] for p in properties]
min_allowed_class_samples=30
#####################################################

df.dropna(inplace=True,how='all', subset = properties)
df.reset_index(drop=True,inplace=True)

model_param['Properties']=properties
model_param['Categorical properties?']=categorical

if not categorical:
    if class_values is not None:
        model_param['Provided class cutoffs']=class_values
    elif class_quantiles is not None:
        model_param['Provided class quantile cutoffs']=class_quantiles

from automol.property_prep import ClassBuilder

if class_values is not None and class_quantiles is not None:
    print('Both class_values and class_quantiles are set, using values')
    class_quantiles=None
    
if class_quantiles is not None:
    class_values=class_quantiles
    
prop_builder=ClassBuilder(properties=properties,nb_classes=nb_classes,class_values=class_values,
                         categorical=categorical,use_quantiles=class_quantiles is not None,
                         prefix='Class',min_allowed_class_samples=30,verbose=verbose)

#check properties
prop_builder.check_properties(df)        
#prepare properties: e.g. create classes, apply transformations etc..c 
df,train_properties=prop_builder.generate_train_properties(df)
#retrieve labelnames 
labelnames=prop_builder.labelnames

test_df,_=prop_builder.generate_train_properties(test_df)

df_smiles=df[standard_smiles_column]
df.head()

### Sample Weights
set **use_sample_weight** to True if you would like assign more weight to a certain class. The class for each property is set in the dictionary **weighted_samples_index**. 
* In the case of categorical data, use the label. 
* If continuous data is fitted, use the class index, see labelnames (the index for class with the smallest values is 0). 

The weight for each property can be set in **select_sample_weights**. The weights are set in the dataframe with the prefix *sw*. Note that it is possible to directly set the column *sw_\<prop\>* with prop the property name as sample weights for property *prop*. 

In [ ]:
#####################################################
#set to true when planning to use sample_weights, some methods do not have this option
use_sample_weight=True
#set class index for non-categorical data, in this case always the class with the smallest values
weighted_samples_index={p:0 for i,p in enumerate(train_properties)}
#set label for categorical data
if categorical:
    weighted_samples_index={p:'0.0' for p in train_properties}
#set weight of the class specified in property_wclass, other classes have weight 1.0
select_sample_weights={p:10 for p in train_properties}
#####################################################

if use_sample_weight: model_param['Provided samples to weighted per property']=weighted_samples_index
if use_sample_weight: model_param['Provided weights per property']=select_sample_weights

if use_sample_weight:
    df=prop_builder.generate_sample_weights(df,weighted_samples_index,select_sample_weights)
df.head()

### Creating the feature generator and model trainer
The class *ModelAndParams* is an interface class to retrieve the model, used method and the parameters to be optimized using default values.

The **task** variable in the case of classification is *Classification* or *clf*. Note that regression is the default.

The **computation_load** variable determines the lay-out and methods used with increased computational workload. 
* *cheap*: a list of base estimators are found and given to a final estimator. The methods used for base estimators are: Logistic Regression, K-nearest neighbors, support vector machines and linear discriminant analysis. The final estimator:Logistic Regression (due to platt scaling of svm for threshold tuning). No dimension reduction techniques are used. Gridsearch is performed to find the best parameters of the methods and the final estimator. 
* *moderate*: a list of base estimators are found and given to a final estimator. The methods used for base estimators are: Logistic Regression, K-nearest neighbors, support vector machines , stochastic gradient descent classifier and light gradient boosting machine classifier. The choice of final estimator is between: Logistic Regression, support vector machines and light gradient boosting machine classifier. No dimension reduction techniques are used. The parameters are optimized using randomizedsearch with 100 iterations. 
* *expensive*: The base estimators are fitted and given to sklearn StackingClassifier is fitted choosing from provided the final estimators. A stacking regressor uses internal cross-validation to fit the final estimator.  The methods used for base estimators are: Logistic Regression, K-nearest neighbors, support vector machines , stochastic gradient descent classifier, decision tree classifier and light gradient boosting machine classifier (4 times!). The choice of final estimator is between: Logistic Regression, support vector machines and light gradient boosting machine classifier. No dimension reduction techniques are used. The parameters are optimized using HyperOpt with 100 iterations. *From the Sklearn documentation: Note that estimators_ are fitted on the full X while final_estimator_ is trained using cross-validated predictions of the base estimators using cross_val_predict. (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)* 

See the expert notebooks for more general use.


The variable **scorer** is the metric used when optimizing the parameters.

In [ ]:
#####################################################
#Classification, RegressionClassification or Regression
task='Classification'
#Estimate of the computational workload, more expensive will take more time, but is not gauranteed to provide better results, although more likely
#choose between cheap, moderate or expensive
computional_load='cheap'
#The metric that is optimized when searching the method parameters, examples:
#set scorer : accuracy, balanced_accuracy, f1_weighted, precision, neg_log_loss, roc_auc_ovr_weighted, recall
scorer='balanced_accuracy'

#random state for internal randomness, e.g. randomizedsearch, hyperopt, ...
random_state=5
#random state values for methods, available as a parameter 
random_state_list=[1,7,42,55,3]
#number of jobs used when searching the optimal method parameters (-1 for all)
n_jobs=20
#####################################################
from automol.stacking_util import ModelAndParams

model_param['Computational power']=computional_load
model_param['Task']=task
model_param['Score function']=scorer
model_param['Random state']=random_state
model_param['Random state list']=random_state_list

#sample_weight not available for expensive option
if computional_load=='expensive':
    use_sample_weight=False


param_Factory=ModelAndParams(task=task,
                             computional_load=computional_load,
                             distribution_defaults=False, #use distributions for parameter selection in randomized search
                             use_gpu=False,
                             normalizer=True, #normalize input base estimators
                             top_normalizer=True,#normalize output base estimators and thus input top estimator
                             random_state=random_state_list,#random_state
                             use_sample_weight=use_sample_weight,
                             verbose=verbose,
                             n_jobs_dict={'outer_jobs': 4,
                                        'inner_jobs': 4,
                                        'method_jobs': 4},
                             labelnames=labelnames)
stacked_model, prefixes,params_grid,blender_params,paramsearch = param_Factory.get_model_and_params()

### Splitting the data in Training and Validation sets
The smiles are clustered and split into a training and a validation set. 

It is possible to choose between the following clustering methods by setting the variable **val_clustering** to:
* *Scaffold*: performs MurckoScaffold clustering and uses the boolean **val_include_chirality**
* *Butina*: performs Butina clustering based on the variable **val_butina_cutoff** and afterwards the smiles are assigned to the closest centroid and possibly not the centroid they were assigned to in the first place. 
* *HierarchicalButina*: Does applies butina clustering hierarchically, here the variable **val_butina_cutoff** should contain a list of increasing cutoff values.
* *Bottleneck*: performs Kmeans++ on the generated Bottleneck features. The value of K and thus the number of cluster is fixed by the variable **val_km_groups**

Validation **strategy**:
* *mixed*: 30% activity or property cliffs, 30% leave group out and rest stratified. Values can be altered.
* *leave group out*: full clusters in validation set.
* *stratified*: samples from each cluster are added to the validation set.

The variable **minority_nb** is the threshold to define clusters as small. If a cluster contains less samples than this value, this cluster added to the cluster *minorities*. This only happens for the stratified and mixed validation strategies.

Select the used feature for kmeans from the list below in **val_used_features_km**. Note that there are over 10k MELLODDY features and use of these features will consume a lot of computational power even when the cheap computational load has been chosen. 

In [ ]:
print(f'available feature generators: {stacked_model.feature_generators.keys()}')

In [ ]:
#####################################################
#Clustering used when dividing data in training and validation
#choose between 'Bottleneck','Butina','HierarchicalButina' or 'Scaffold'
val_clustering='Bottleneck'
# include chirality for MurckoScaffolds
val_include_chirality=False
#value of k for Kmeans++, e.g. number of clusters
val_km_groups=30
#select used features in kmeans
val_used_features_km=['Bottleneck']
if val_clustering=="Butina":
    #cutoff value for butina clustering
    val_butina_cutoff=0.5
else:
    #list of cutoffs for hierarchical butina clustering (increase the ratio)
    val_butina_cutoff=[0.2,0.4,0.6]  
#mixed, stratified or leave_group_out
strategy='mixed'
# ratio of size validation set/full data
test_size=0.25
# Small clusters are grouped in one larger cluster for stratified validation (and thus also mixed). 
# This value is the threshold to determine which clusters are 'small'
minority_nb=5
#####################################################

from automol.validation import stratified_validation, leave_grp_out_validation, mixed_validation
from automol.stacking_util import get_clustering_algorithm

model_param['Clustering algorithm used for validation']=val_clustering
model_param['Validation set strategy used']=strategy
model_param['Ratio of test size wrt to total data size']=test_size
model_param['minority_nb']=minority_nb
if val_clustering=="Scaffold" : model_param['Include chiralty?']=val_include_chirality
if val_clustering=="Butina" or val_clustering=="HierarchicalButina": model_param['Threshold(s) used for butina when creating Validation set']=val_butina_cutoff
if val_clustering=="Bottleneck" : 
    model_param['Number of clusters for Kmeans++']=val_km_groups    
    model_param['Used features for Kmeans++']=val_used_features_km

c_algo=get_clustering_algorithm(clustering=val_clustering,
                             n_clusters=val_km_groups,
                             cutoff=val_butina_cutoff,
                             include_chirality=val_include_chirality,
                             verbose=verbose,
                             random_state=random_state,
                             feature_generators=stacked_model.feature_generators,
                             used_features=val_used_features_km)


#split the data in training and validation
groups_left_out=None
prop_cliff_dict=None
if strategy=='mixed':
    #butina threshold for activity cliffs
    prop_cliff_butina_th=0.45
    # property difference between values p1 and p2 is considered an activity cliff if abs(p2-p1)>rel_prop_cliff * (max(p_i)-min(p_i))
    rel_prop_cliff=0.5
    #stratified is essentially the remaining number of samples left to reach the desired amount of test samples set by test_size ratio
    #    and is therefore not actually used
    mix_coef_dict={ 'prop_cliffs': 0.3,'leave_group_out': 0.3 ,'stratified': 0.4}
    if categorical:
        #restricting threshold for categorical data
        prop_cliff_butina_th=0.35
        print('Activity cliffs with categorical properties: similar compounds with different class are considered as an activity cliff ')
    Train, Validation,groups_left_out, prop_cliff_dict = mixed_validation(df_orig=df,properties=properties,stacked_model=stacked_model,standard_smiles_column=standard_smiles_column,
                                                  prop_cliff_cut=rel_prop_cliff,prop_cliff_butina=prop_cliff_butina_th,test_size=test_size,
                                                  verbose=verbose,random_state=random_state,mix_dict=mix_coef_dict, categorical_data=categorical,
                                                  minority_nb=minority_nb,clustering_algorithm=c_algo)
    
elif strategy=='stratified':
    Train, Validation = stratified_validation(df,train_properties,stacked_model,standard_smiles_column,df_smiles,test_size=test_size,
                                                  verbose=verbose,random_state=random_state,
                                                  minority_nb=minority_nb,clustering_algorithm=c_algo)
    
else:
    Train, Validation = leave_grp_out_validation(df,train_properties,stacked_model,standard_smiles_column,df_smiles,test_size=test_size,
                                                  verbose=verbose,random_state=random_state,clustering_algorithm=c_algo)
    groups_left_out=np.arange(len(Validation))

#placing the validation and training set inside the model generator
stacked_model.Validation=Validation
stacked_model.Train=Train
stacked_model.smiles=standard_smiles_column # set col smiles name

### Clustering the data for nested cross-validation
same options as for clustering the data for splitting in training and validation sets

In [ ]:
#####################################################
#Cluster the data within the models into groups to be used for NCV 
#Clustering used when defining groups in nested cross-validation to select parameters of models
#choose between 'Bottleneck','Butina' or 'Scaffold' with Bottleneck: kmeans on the generated features by the Bottleneck model
cv_clustering='Bottleneck'
# include chirality for MurckoScaffolds
include_chirality=False
#value of k for Kmeans++, e.g. number of clusters
km_groups=20
#used features for kmeans
used_features_km=['Bottleneck']
if cv_clustering=="Butina":
    #cutoff value for butina clustering
    butina_cutoff=0.6
else:
    #list of cutoffs for hierarchical butina clustering (increase the ratio)
    butina_cutoff=[0.2,0.4,0.6]
#####################################################

model_param['Clustering algorithm used for Nested cross-validation']=cv_clustering
if cv_clustering=="Scaffold" : model_param['Include chiralty for nested cross-validation?']=include_chirality
if cv_clustering=="Butina" or val_clustering=="HierarchicalButina": model_param['Threshold(s) used for butina when clustering for nested cross-validation']=butina_cutoff
if cv_clustering=="Bottleneck" : 
    model_param['Number of clusters for Kmeans++ for Nested cross-validation']=km_groups
    model_param['Used features for Kmeans++']=used_features_km

cv_c_algo=get_clustering_algorithm(clustering=cv_clustering,
                             n_clusters=km_groups,
                             cutoff=butina_cutoff,
                             include_chirality=include_chirality,
                             verbose=verbose,
                             random_state=random_state,
                             feature_generators=stacked_model.feature_generators,
                             used_features=used_features_km)

#property check
for p in properties:
    prop_count=df[p].count()
    if cv_clustering=='Bottleneck' and prop_count/km_groups<10:
        print('Warning: on average less than 10 samples per cluster for property',p,', suggested use is to decrease number of groups')
        
# cluster the data within the models into groups to be used for NCV 
stacked_model.Data_clustering(random_state=random_state, clustering_algorithm=cv_c_algo)

### Searching models for the properties using cross-validation
The different methods are fitted using the chosen parameters above.

The cross-validation is determined by the parameter **outer_folds**, e.g. the number of outer folds (number of inner folds is outer - 1) and the **cross_val_split**. Choose for the latter between *LGO*: leave one group out, *GKF*: group k-fold which splits full groups in cross-validation manner or *SKF*: stratified K-fold which makes sure that the split preserves the ratio of samples between classes (only recommended when the number of samples is really low for one class, try GKF first and if GKF fails try SKF). 

This way, for each fold the base estimators are optimized for a specific chemical subspace in the inner folds. The outer folds are then used to fit final estimator based on the output of the base estimators. 


In [ ]:
%%time  
#####################################################
#cross-validation parameters
#select cross-validation split:  LGO, SKF or GKF
cross_val_split='GKF'
outer_folds=4
#select the features used when training the model
used_features=['Bottleneck']
#####################################################
model_param['Used cross-validation technique']=cross_val_split
if cross_val_split != 'LGO': model_param['Number of folds']=outer_folds

if outer_folds>km_groups:
    if km_groups>2:
        outer_folds=km_groups-1
    else:
        km_groups=outer_folds+1
        
results_dictionary={}
#empty to put this one first (keep in mind order in dictionary is not reliable)
for p in train_properties:
    results_dictionary[p]={}      
    sample_weight=None
    if use_sample_weight:
        sample_weight=stacked_model.Train[f'sw_{p}'].values
    stacked_model.search_model(df= None,   prop=p,  smiles=standard_smiles_column,
                             params_grid=params_grid,
                               paramsearch=paramsearch,
                              scoring=scorer,#'neg_mean_absolute_error',#
                              cv=outer_folds-1,  outer_cv_fold=outer_folds, split=cross_val_split, 
                              use_memory=True,
                              features=used_features,
                              plot_validation=True, 
                             refit=False,# no refit with validation. comes later,
                             blender_params=blender_params
                              ,prefix_dict=prefixes,random_state=random_state,sample_weight=sample_weight
                              ,results_dict=results_dictionary)
model_str=stacked_model.print_metrics()

### Validation
show the results on the validation set. The variable **cmap** is a list of colormaps used for the figures.

In [ ]:
#####################################################
#set colormap for classification report and confusion matrix: PiYG, viridis, YlGnBu, Blues
cmap=['PiYG','Blues']
#####################################################
from automol.stacking_util import ResultDesigner

out=stacked_model.predict( props =None, smiles=stacked_model.Validation[standard_smiles_column],compute_SD=True,convert_log10=False)

print('##########################################################')
print('                    Validation set')
youden_dict=ResultDesigner().show_classification_report(train_properties,out,[stacked_model.Validation[f'{p}'].values for p in train_properties],labelnames=labelnames,cmap=cmap)
F1_dict=ResultDesigner().show_clf_threshold_report(train_properties,out,[stacked_model.Validation[f'{p}'].values for p in train_properties],labelnames=labelnames,youden_dict=youden_dict)

### Test set
show the results on the Test set.  

In [ ]:
#####################################################
#set colormap for classification report and confusion matrix: PiYG, viridis, YlGnBu, Blues
cmap=['PiYG','Blues']
#####################################################
from automol.stacking_util import ResultDesigner

out=stacked_model.predict( props =None, smiles=test_df[standard_smiles_column],compute_SD=True,convert_log10=False)

print('##########################################################')
print('                    Validation set')
youden_dict=ResultDesigner().show_classification_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,cmap=cmap)
F1_dict=ResultDesigner().show_clf_threshold_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,youden_dict=youden_dict)

### Generate plotly figures

The variable **show_plotly_figures** can be set to False if you experience 403 errors when trying to save the notebook (For certain browsers this can happen, clear the output from this cell and the notebook wil save).

The variables **show_adv** and **show_th** can be set to display advanced classifier figures and threshold tuning figures respectively

In [ ]:
#import plotly.offline as py
######################################################
show_plotly_figures=True
show_adv=True
show_th=True
x_width=600
y_height=600
#######################################################
from automol.plotly_util import PlotlyDesigner
from automol.plotly_util import create_figure_captions, get_figures_from_types
youden_dict,fig_l=PlotlyDesigner().show_classification_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],
                                                              labelnames=labelnames,cmap=cmap,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)
fig_adv=PlotlyDesigner().show_additional_classification_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],
                                                               labelnames=labelnames,cmap=cmap,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)
F1_dict,fig_th=PlotlyDesigner().show_clf_threshold_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],youden_dict=youden_dict,
                                                          labelnames=labelnames,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)

plotly_dictionary={**fig_l,**fig_th,**fig_adv}
plotly_show={**fig_l}
if show_th:    
    plotly_show={**plotly_show,**fig_th}
if show_adv:
    plotly_show={**plotly_show,**fig_adv}


_,_,type_keys=create_figure_captions(plotly_dictionary.keys())
#retrieve figure keys from types ordered per property
sorted_keys=get_figures_from_types(plotly_dictionary=plotly_show,properties=train_properties,types=type_keys)
#retrieve captions in order of sorted keys
captions,types,_=create_figure_captions(sorted_keys)

if show_plotly_figures:
    for index,key in enumerate(sorted_keys):
        plotly_dictionary[key].show()
        print(captions[index])


### Generate pdf summary
Generate a small report for the model. You can set the authors by defining **authors**, the title by **title**, contact information in **mails** and add a small summary in **summary**. You can choose the types of figures to be added to the report in **figures_types** as well as the number of columns for these figures in **nb_columns**. The name of the pdf file can be set in **output_file**.  

You can provide your own template in **template_file**, if this variable is None, the template of the package is used. You may tweak summary.html to your satisfaction, provided in the tutorial directory. 

In [ ]:
print('Figure types:\n')
for it,t in enumerate(types):
    print(f"{it+1}. "+t+"\n")

In [ ]:
#####################################################
output_file="summary.pdf"
title='AutoMoL report Title'
authors='Joris Tavernier'
mails='joris.tavernier@openanalytics.eu'
summary="""Project details -- Heri, inquam, ludis commissis ex urbe profectus veni ad vesperum.
causa autem fuit huc veniendi ut quosdam hinc libros promerem. et
quidem, Cato, hanc totam copiam iam Lucullo nostro notam esse oportebit;
nam his libris eum malo quam reliquo ornatu villae delectari. est enim
mihi magnae curae - quamquam hoc quidem proprium tuum munus est - ut ita
erudiatur, ut et patri et Caepioni nostro et tibi tam propinquo
respondeat. laboro autem non sine causa; nam et avi eius memoria moveor
- nec enim ignoras, quanti fecerim Caepionem, qui, ut opinio mea fert,
in principibus iam esset, si viveret - et Lucullus mihi versatur ante
oculos, vir cum virtutibus omnibus excellens, tum mecum et amicitia et
omni voluntate sententiaque coniunctus."""
#use self provided template from notebook directory
#template_file="summary.html"
#Set template_file to None to use package template
template_file=None
#select figure types to be added to the report
figure_types=['A','B','H','I','M']
nb_columns=3
#####################################################
from automol.plotly_util import generate_report, get_figures_from_types

#retrieve figure keys from types ordered per property
selected_figures=get_figures_from_types(plotly_dictionary=plotly_dictionary,properties=train_properties,types=figure_types)

#you can adjust selected_figures before this statement to subselect figures within a type
captions,types,_=create_figure_captions(selected_figures)
pdf_data=generate_report(plotly_dictionary,selected_figures,captions,types,title=title, authors=authors,
                         mails=mails,summary=summary,model_param=model_param,model_tostr_list=model_str,properties=train_properties,template_file=template_file,nb_columns=nb_columns )
with open(output_file, mode="wb") as f:
    f.write(pdf_data)

### Saving the model(s)
the models wil be saved in **save_model_to_file**. The model is refitted on both the training and validation set. The model(s) will additionally be automatically and internally evaluated on a few public smiles for reproducability, which can be verified when loading the model.  

In [ ]:
from automol.plotly_util import save_result_dictionary_to_json
#####################################################
#the filename where the model will be saved
json_file='results_dictionary.json' 
yaml_file='training_param.yaml'
#####################################################
save_result_dictionary_to_json(result_dict=results_dictionary,json_file=json_file)
#import yaml
#with open(yaml_file, 'w') as file:
#    documents = yaml.dump(model_param, file)


In [ ]:
#####################################################
#the filename where the model will be saved
save_model_to_file='Cypro_stackingclfmodel.pt'
#####################################################
from automol.stacking import save_model


for p in train_properties:
    sample_train=None
    sample_val=None
    if use_sample_weight:
        sample_train=stacked_model.Train[f'sw_{p}'].values
        sample_val=stacked_model.Validation[f'sw_{p}'].values
    stacked_model.refit_model(models=p,sample_train=sample_train,sample_val=sample_val,prefix_dict=prefixes)
    
## clean the class first by removing the computed features
stacked_model.clean()
stacked_model.compute_SD=True
save_model(stacked_model, save_model_to_file)
stacked_model.validate(df=None, # df with smiles and the properties
                       props=None, # name of the task 
                    true_props=None,# name of the property in df
                    smiles=standard_smiles_column)

### Test set results after refitting

In [ ]:
#####################################################
#set colormap for classification report and confusion matrix: PiYG, viridis, YlGnBu, Blues
cmap=['PiYG','Blues']
#####################################################
from automol.stacking_util import ResultDesigner

out=stacked_model.predict( props =None, smiles=test_df[standard_smiles_column],compute_SD=True,convert_log10=False)

print('##########################################################')
print('                    Validation set')
youden_dict=ResultDesigner().show_classification_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,cmap=cmap)
F1_dict=ResultDesigner().show_clf_threshold_report(train_properties,out,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,youden_dict=youden_dict)

### Loading the model
The reproducable output is automatically verified and the maximum error for each property is printed if the check fails. If not a succes message is printed.  

In [ ]:
from automol.stacking import load_model

stacked_model2= load_model( save_model_to_file,use_gpu=False) 
stacked_model2.print_metrics()

### Saving the notebook and the output as html

Save the notebook first and then execute the cell. This will save this notebook, set the name in variable **notebook_name** without the .ipynb extension. The html will then be saved in **save_output_to_file** 

In [ ]:
#####################################################
#the name of this notebook, is important for saving the output (without the .ipynb)
notebook_name='Classifier'
#html file where the output is
save_output_to_file='Cypro_stackingclfmodel.html'
#####################################################
notebook_file=notebook_name+'.ipynb'
notebook_html=notebook_name+'.html'
!jupyter-nbconvert --to html $notebook_file
!mv $notebook_html $save_output_to_file

In [ ]:
results_dictionary